# Clase 175 — Distribuciones: normal, binomial, Poisson, exponencial

Las cuatro distribuciones que aparecen en el 90 % de los problemas reales: **normal** (sumas/promedios por TCL), **binomial** (éxitos/fracasos), **Poisson** (conteos raros) y **exponencial** (tiempos entre eventos). Simulamos, comparamos contra la PDF/PMF teórica y verificamos con Q-Q plot + Kolmogorov-Smirnov.

Requiere: `numpy`, `scipy`, `matplotlib`.

## 1. Distribución normal N(μ, σ²)

Simétrica, soporte en toda la recta. Base de t-test, ANOVA e intervalos de confianza. Verificamos la regla 68-95-99.7.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
np.random.seed(42)

mu, sigma = 100.0, 15.0
x = rng.normal(mu, sigma, 10_000)
print(f"Normal N(100, 15^2): media={x.mean():.2f}  sd={x.std(ddof=1):.2f}")

for k in (1, 2, 3):
    frac = np.mean(np.abs(x - mu) < k * sigma)
    print(f"  dentro de {k} sigma: {frac:.3f}")
frac1 = np.mean(np.abs(x - mu) < sigma)
assert abs(frac1 - 0.683) < 0.02, "la regla 68-95-99.7 debe cumplirse"

grid = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
plt.figure(figsize=(7, 4))
plt.hist(x, bins=50, density=True, alpha=0.5, label="muestra")
plt.plot(grid, stats.norm(mu, sigma).pdf(grid), "r-", lw=2, label="PDF teórica")
plt.title("Distribución normal: histograma vs PDF"); plt.legend()
plt.tight_layout(); plt.show()

## 2. Distribución binomial Bin(n, p)

Suma de `n` ensayos Bernoulli. `E[X]=np`, `Var[X]=np(1-p)`. Modela conversiones y proporciones.

In [ ]:
n_trials, p = 20, 0.3
b = rng.binomial(n_trials, p, 10_000)
print(f"Binomial(20, 0.3): E[X]={b.mean():.3f} (teórico {n_trials*p})  "
      f"Var={b.var(ddof=1):.3f} (teórico {n_trials*p*(1-p)})")
assert abs(b.mean() - n_trials*p) < 0.2

k = np.arange(0, n_trials + 1)
pmf = stats.binom(n_trials, p).pmf(k)
plt.figure(figsize=(7, 4))
plt.hist(b, bins=np.arange(-0.5, n_trials + 1.5), density=True, alpha=0.5, label="muestra")
plt.plot(k, pmf, "ro-", label="PMF teórica")
plt.title("Binomial(20, 0.3): conteo de éxitos"); plt.legend()
plt.tight_layout(); plt.show()

## 3. Distribución de Poisson Poi(λ)

Conteo de eventos raros por unidad de tiempo. Propiedad clave: `E[X]=Var[X]=λ` (equidispersión). Si `Var/Media ≫ 1` hay **sobredispersión** y Poisson no aplica.

In [ ]:
lam = 4.2
pois = rng.poisson(lam, 10_000)
ratio = pois.var(ddof=1) / pois.mean()
print(f"Poisson(4.2): media={pois.mean():.3f}  var={pois.var(ddof=1):.3f}  var/media={ratio:.3f}")
print("Equidispersión (var≈media) => Poisson plausible" if 0.9 < ratio < 1.1 else "sospecha de sobre/infra-dispersión")
assert 0.9 < ratio < 1.1

cont = np.concatenate([pois, rng.poisson(20, 400)])
print(f"Contaminado con eventos grandes: var/media={cont.var(ddof=1)/cont.mean():.3f} "
      f"(>1 => sobredispersión, Poisson NO aplica)")
assert cont.var(ddof=1) / cont.mean() > 1.2

## 4. Distribución exponencial Exp(λ)

Tiempo entre eventos Poisson. Es **memoryless**: `P(X > s+t | X > s) = P(X > t)`.

In [ ]:
e = rng.exponential(1.0, 100_000)   # scale = 1/λ con λ=1
print(f"Exponencial(λ=1): media={e.mean():.3f} (teórica 1.000)")

s, t = 1.0, 0.5
cond = np.mean(e[e > s] > s + t)
marg = np.mean(e > t)
print(f"P(X>1.5 | X>1) = {cond:.3f}   vs   P(X>0.5) = {marg:.3f}")
assert abs(cond - marg) < 0.02, "la exponencial es memoryless" 

## 5. Teorema Central del Límite

La media de variables i.i.d. (aunque no sean normales) tiende a normal al crecer `n`. Promediamos `Exp(1)` con distintos `n` y medimos cómo la asimetría cae hacia 0.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
skews = {}
for ax, n_avg in zip(axes, (1, 5, 30, 100)):
    medias = rng.exponential(1.0, (5000, n_avg)).mean(axis=1)
    skews[n_avg] = stats.skew(medias)
    ax.hist(medias, bins=40, density=True, color="steelblue", alpha=0.7)
    ax.set_title(f"n={n_avg}  (asim={skews[n_avg]:.2f})")
plt.suptitle("TCL: la media de Exp(1) se vuelve normal al crecer n")
plt.tight_layout(); plt.show()

print(f"asimetría de la media: n=1 -> {skews[1]:.2f},  n=100 -> {skews[100]:.2f}")
assert abs(skews[100]) < abs(skews[1])

## 6. Verificación empírica: Q-Q plot + KS test

Antes de asumir una distribución, mirá. Comparamos un `total_bill` sintético (asimétrico positivo) contra normal y contra lognormal.

In [ ]:
total_bill = rng.lognormal(mean=2.8, sigma=0.4, size=300)   # imita 'tips'

z = (total_bill - total_bill.mean()) / total_bill.std(ddof=1)
ks_norm = stats.kstest(z, "norm")
shape, loc, scale = stats.lognorm.fit(total_bill, floc=0)
ks_logn = stats.kstest(total_bill, "lognorm", args=(shape, loc, scale))
print(f"KS vs normal:     stat={ks_norm.statistic:.3f}  p={ks_norm.pvalue:.4f}")
print(f"KS vs lognormal:  stat={ks_logn.statistic:.3f}  p={ks_logn.pvalue:.4f}")
assert ks_logn.pvalue > ks_norm.pvalue, "lognormal ajusta mejor que normal"

fig, ax = plt.subplots(figsize=(6, 5))
stats.probplot(total_bill, dist="norm", plot=ax)
ax.set_title("Q-Q plot vs normal: la cola derecha se despega")
plt.tight_layout(); plt.show()

## Ejercicios

1. Calculá `stats.norm(100, 15).ppf([0.025, 0.5, 0.975])` y verificá que los percentiles 2.5 % y 97.5 % de una simulación con `n=100_000` se aproximan a esos cuantiles teóricos.
2. Repetí el chequeo de equidispersión con `lam=10` y con una mezcla de dos Poisson; confirmá que el ratio `var/media` detecta la sobredispersión.
3. Cambiá el Q-Q plot a `dist="lognorm"` con `sparams=(shape,)` y observá cómo los puntos caen sobre la diagonal.

## Conclusiones

- Cada distribución modela un fenómeno concreto: normal (promedios por TCL), binomial (éxitos/fracasos), Poisson (conteos raros), exponencial (tiempos entre eventos).
- La equidispersión `var≈media` es el test diagnóstico de Poisson; si `var/media ≫ 1` hay sobredispersión (usar binomial negativa).
- El TCL explica por qué la media muestral se vuelve normal aunque los datos crudos no lo sean.
- Antes de asumir normalidad, verificá con Q-Q plot y KS/Shapiro sobre datos estandarizados.

## ✅ Soluciones de los ejercicios
<!--SOL175184-->

Soluciones trabajadas y **ejecutables** de todos los ejercicios de la sección *🧪 Ejercicios*. Datos sintéticos reproducibles con `np.random.default_rng(42)`; sin dependencias de internet. Cada bloque incluye `assert`/`print` para autocorregir.

<!--SOL175184-->

**Ej. 1 — Simulación vs PDF/PMF teórica.** 10 000 muestras de cada distribución con su densidad/masa teórica de `scipy.stats` superpuesta.

In [ ]:
# <!--SOL175184-->
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 10_000
norm_s = rng.normal(100, 15, N)
binom_s = rng.binomial(20, 0.3, N)
pois_s = rng.poisson(4.2, N)
expo_s = rng.exponential(1.0, N)

fig, ax = plt.subplots(1, 4, figsize=(14, 3))
xs = np.linspace(norm_s.min(), norm_s.max(), 200)
ax[0].hist(norm_s, bins=40, density=True, alpha=.5); ax[0].plot(xs, stats.norm(100, 15).pdf(xs), lw=2); ax[0].set_title("Normal(100,15)")
k = np.arange(0, 21)
ax[1].hist(binom_s, bins=np.arange(-.5, 21.5), density=True, alpha=.5); ax[1].plot(k, stats.binom(20, 0.3).pmf(k), "o-", lw=1); ax[1].set_title("Binomial(20,0.3)")
kp = np.arange(0, pois_s.max() + 1)
ax[2].hist(pois_s, bins=np.arange(-.5, pois_s.max() + 1.5), density=True, alpha=.5); ax[2].plot(kp, stats.poisson(4.2).pmf(kp), "o-", lw=1); ax[2].set_title("Poisson(4.2)")
xe = np.linspace(0, expo_s.max(), 200)
ax[3].hist(expo_s, bins=40, density=True, alpha=.5); ax[3].plot(xe, stats.expon(scale=1.0).pdf(xe), lw=2); ax[3].set_title("Exp(1)")
fig.tight_layout()

assert abs(norm_s.mean() - 100) < 1.0
assert abs(binom_s.mean() - 6.0) < 0.2
assert abs(pois_s.mean() - 4.2) < 0.15
assert abs(expo_s.mean() - 1.0) < 0.05
print("Medias muestrales ~ teoricas: OK")

<!--SOL175184-->

**Ej. 2 — Cuantiles.** IC 95 % poblacional del IQ y verificación empírica con `n=100_000`.

In [ ]:
# <!--SOL175184-->
iq = stats.norm(loc=100, scale=15)
q = iq.ppf([0.025, 0.5, 0.975])
sim = rng.normal(100, 15, 100_000)
emp = np.quantile(sim, [0.025, 0.5, 0.975])
print(f"teoricos  = {q.round(2)}")
print(f"empiricos = {emp.round(2)}")
assert np.allclose(q, emp, atol=0.5)
assert abs(q[1] - 100) < 1e-9
print("Cuantiles teoricos ~ empiricos: OK")

<!--SOL175184-->

**Ej. 3 — TCL en acción.** Promedios de `Exp(1)` con `n` creciente: la media muestral se vuelve simétrica.

In [ ]:
# <!--SOL175184-->
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
sk = []
for j, n in enumerate([1, 5, 30, 100]):
    medias = rng.exponential(1.0, size=(5_000, n)).mean(axis=1)
    sk.append(stats.skew(medias))
    ax[j].hist(medias, bins=40, density=True, alpha=.6); ax[j].set_title(f"n={n} skew={sk[-1]:.2f}")
fig.suptitle("Medias de Exp(1): el TCL en accion"); fig.tight_layout()
print("skew por n:", [round(s, 2) for s in sk])
assert sk[0] > sk[-1]
assert abs(sk[-1]) < 0.4
print("El TCL simetriza la media muestral: OK")

<!--SOL175184-->

**Ej. 4 — Q-Q plot** de una `total_bill` sintética lognormal (acotada en 0, cola derecha).

In [ ]:
# <!--SOL175184-->
total_bill = rng.lognormal(mean=2.8, sigma=0.4, size=300)
fig, ax = plt.subplots(figsize=(5, 4))
(osm, osr), (slope, inter, r) = stats.probplot(total_bill, dist="norm", plot=ax)
ax.set_title("Q-Q plot total_bill vs Normal")
resid_der = osr[-10:] - (slope * osm[-10:] + inter)
print(f"skew={stats.skew(total_bill):.2f}  residuo cola derecha={resid_der.mean():.2f}")
assert stats.skew(total_bill) > 0.3 and resid_der.mean() > 0
print("Q-Q confirma asimetria positiva: OK")

<!--SOL175184-->

**Ej. 5 — ¿Poisson o no?** Equidispersión (`Var/Mean≈1`) vs sobre-dispersión por contaminación.

In [ ]:
# <!--SOL175184-->
puro = rng.poisson(4.2, 10_000)
ratio_puro = puro.var() / puro.mean()
contaminado = np.concatenate([puro, rng.poisson(20, 200)])
ratio_cont = contaminado.var() / contaminado.mean()
print(f"puro Var/Mean={ratio_puro:.3f}   contaminado Var/Mean={ratio_cont:.3f}")
assert 0.8 <= ratio_puro <= 1.2 and ratio_cont > 1.2
print("Equidispersion vs sobre-dispersion: OK")